<a href="https://colab.research.google.com/github/MMASirOwlbie/CNNProject-StarDetectionReproduction/blob/main/ML_Project_Milestone_2_Star_Detection_Training_and_Reproduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

%cd /content
!rm -rf CNNStarDetectCentroid
!git clone https://github.com/HongruiZhao/CNNStarDetectCentroid.git
%cd /content/CNNStarDetectCentroid

!pip install thop -q

# code patches
with open("training/neural_net/CentroidNet.py", "r") as f:
    code = f.read()
with open("training/neural_net/CentroidNet.py", "w") as f:
    f.write(code.replace("from numpy.lib.arraypad import pad", "from numpy import pad"))

for script in ["data_generation/main_generate_micro_data.py", "data_generation/main_generate_data.py"]:
    if os.path.exists(script):
        with open(script, "r") as f:
            code = f.read()
        with open(script, "w") as f:
            f.write(code.replace("os.mkdir(", "os.makedirs(exist_ok=True, name="))

#create directories
os.makedirs("/content/CNNStarDetectCentroid/training/runs", exist_ok=True)
os.makedirs("/content/CNNStarDetectCentroid/training/models", exist_ok=True)
os.makedirs("/content/CNNStarDetectCentroid/training/checkpoint", exist_ok=True)
os.makedirs("/content/CNNStarDetectCentroid/data_generation/training_data", exist_ok=True)

#datagen
%cd /content/CNNStarDetectCentroid/data_generation
!gdown 1_uvcg0AcxmWhJoxbO5dH1Y51mUlNPW3R -O dark_frames_straylight.zip
!unzip -q -o dark_frames_straylight.zip

from main_generate_data import star_img_sim, generate_and_save

sim_obj = star_img_sim(500, "./dark_frames_straylight")

print("Generating 500 train images...")
generate_and_save(sim_obj, 300, "train", "./training_data")

print("Generating 100 val images...")
generate_and_save(sim_obj, 100, "val", "./training_data")

print("Generating 100 test images...")
generate_and_save(sim_obj, 100, "test", "./training_data")

print("\Complete!")

/content
Cloning into 'CNNStarDetectCentroid'...
remote: Enumerating objects: 371, done.
remote: Total 371 (delta 0), reused 0 (delta 0), pack-reused 371 (from 2)
Receiving objects: 100% (371/371), 232.64 MiB | 16.02 MiB/s, done.
Resolving deltas: 100% (92/92), done.
Updating files: 100% (63/63), done.
/content/CNNStarDetectCentroid
/content/CNNStarDetectCentroid/data_generation
Downloading...
From (original): https://drive.google.com/uc?id=1_uvcg0AcxmWhJoxbO5dH1Y51mUlNPW3R
From (redirected): https://drive.google.com/uc?id=1_uvcg0AcxmWhJoxbO5dH1Y51mUlNPW3R&confirm=t&uuid=2b409ee1-56d4-4c9a-a8dd-6f2c92ce9628
To: /content/CNNStarDetectCentroid/data_generation/dark_frames_straylight.zip
100% 408M/408M [00:09<00:00, 42.5MB/s]
num of dark frames currently in folder = 1000
Generating 500 train images...
current progress = 1 out of 300. time = 6.84 s
current progress = 2 out of 300. time = 3.96 s
current progress = 3 out of 300. time = 6.38 s
current progress = 4 out of 300. time = 4.95 s
cur

In [ ]:
import os

%cd /content/CNNStarDetectCentroid/training

#patch
with open("training_stepLR.py", "r") as f:
    code = f.read()

#help me
patched_code = code.replace(
    'device = torch.device("cuda")',
    'device = torch.device("cuda" if torch.cuda.is_available() else "cpu")'
).replace(
    "device = torch.device('cuda')",
    "device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')"
)

#force-save check since I no longer trust anything
if "latest_model.pt" not in patched_code:
    patched_code += "\n\ntorch.save(model.state_dict(), './models/latest_model.pt')\nprint('Model successfully saved to ./models/latest_model.pt!')\n"

with open("training_stepLR.py", "w") as f:
    f.write(patched_code)

print("Patched training script for device selection!")

!python training_stepLR.py --ep 30 --batch_size 1 --trial 1

/content/CNNStarDetectCentroid/training
✅ Patched training script for dynamic CPU/GPU device selection!
2026-09-27 17:13:05.387285: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-27 17:13:05.589767: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "/content/CNNStarDetectCentroid/training/training_stepLR.py", line 148, in <module>
    model = ELUnet(1,2,8).to(device)
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1384, in to
    return self._apply(convert)
           ~~~~~~~~~~~^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 934, in _apply
    module._apply(

In [ ]:
import glob

pt_files = glob.glob("/content/CNNStarDetectCentroid/**/*.pt", recursive=True)
event_files = glob.glob("/content/CNNStarDetectCentroid/**/*.tfevents*", recursive=True)

print("Checkpoints:", pt_files)
print("TensorBoard Event Logs:", event_files)

#given files are there:
%load_ext tensorboard
%tensorboard --logdir /content/CNNStarDetectCentroid/training/runs

In [ ]:

import os
import torch
import numpy as np
import scipy.spatial
import csv
from torch.utils.data import DataLoader
from data_load import StarDataSet


%cd /content/CNNStarDetectCentroid/training
metrics_path = "/content/CNNStarDetectCentroid/training/test_metrics.csv"

from neural_net.elunet import ELUnet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = ELUnet(in_channels=1, out_channels=2).to(device)

def extract_centroids(pred_binary, pred_offsets):
    y_indices, x_indices = np.where(pred_binary > 0)

    if len(y_indices) == 0:
        return np.empty((0, 2))

    dx = pred_offsets[0, y_indices, x_indices]
    dy = pred_offsets[1, y_indices, x_indices]

    pred_x = x_indices + dx
    pred_y = y_indices + dy

    return np.column_stack((pred_x, pred_y))


def match_star_centroids(pred_coords, gt_coords, dist_threshold=1.5):
    if len(pred_coords) == 0:
        return 0, 0, len(gt_coords)
    if len(gt_coords) == 0:
        return 0, len(pred_coords), 0

    dists = scipy.spatial.distance.cdist(pred_coords, gt_coords)

    pred_indices, gt_indices = np.where(dists <= dist_threshold)
    sorted_pairs = sorted(
        zip(pred_indices, gt_indices), key=lambda pair: dists[pair[0], pair[1]]
    )

    tp = 0
    matched_gt = set()
    matched_pred = set()

    for pred_idx, gt_idx in sorted_pairs:
        if pred_idx not in matched_pred and gt_idx not in matched_gt:
            tp += 1
            matched_pred.add(pred_idx)
            matched_gt.add(gt_idx)

    fp = len(pred_coords) - len(matched_pred)
    fn = len(gt_coords) - len(matched_gt)

    return tp, fp, fn


def compute_test_metrics(model_path, test_loader, dist_threshold=1.5):

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    total_tp, total_fp, total_fn = 0, 0, 0

    with torch.no_grad():
        for images, gt_masks, gt_centroids in test_loader:
            images = images.to(device)
            pred_masks, pred_offsets = model(images)

            pred_binary = (
                (torch.sigmoid(pred_masks) > 0.5).squeeze(1).cpu().numpy()
            )
            pred_offsets_np = pred_offsets.cpu().numpy()

            for i in range(images.size(0)):
                pred_coords = extract_centroids(
                    pred_binary[i], pred_offsets_np[i]
                )

                gt_coords = gt_centroids[i]
                if isinstance(gt_coords, torch.Tensor):
                    gt_coords = gt_coords.cpu().numpy()

                tp, fp, fn = match_star_centroids(
                    pred_coords, gt_coords, dist_threshold
                )
                total_tp += tp
                total_fp += fp
                total_fn += fn

    #Paper Equations (13, 14, 15)
    precision = (
        (total_tp / (total_tp + total_fp) * 100)
        if (total_tp + total_fp) > 0
        else 0.0
    )
    recall = (
        (total_tp / (total_tp + total_fn) * 100)
        if (total_tp + total_fn) > 0
        else 0.0
    )
    f1 = (
        (2 * precision * recall / (precision + recall))
        if (precision + recall) > 0
        else 0.0
    )

    print("=" * 50)
    print("PAPER EVALUATION METRICS")
    print("=" * 50)
    print(f"True Positives (TP)   : {total_tp}")
    print(f"False Positives (FP)  : {total_fp}")
    print(f"False Negatives (FN)  : {total_fn}")
    print("-" * 50)
    print(f"Precision (Eq. 13)    : {precision:.2f}%")
    print(f"Recall    (Eq. 14)    : {recall:.2f}%")
    print(f"F1 Score  (Eq. 15)    : {f1:.2f}%")
    print("=" * 50)

    return precision, recall, f1

test_dataset = StarDataSet(
    split="test",
    data_dir="/content/CNNStarDetectCentroid/data_generation/training_data"
)

def star_collate_fn(batch):
    images = torch.stack([torch.as_tensor(item[0]) for item in batch])
    masks = torch.stack([torch.as_tensor(item[1]) for item in batch])
    centroids = [item[3] if len(item) > 3 else item[2] for item in batch]
    return images, masks, centroids


test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=star_collate_fn
)

model_path = "/content/CNNStarDetectCentroid/training/models/latest_model.pt"
precision, recall, f1 = compute_test_metrics(model_path, test_loader)

with open(metrics_path, "w", newline="") as f:
    writer = csv.writer(f)

    writer.writerow([
        "Precision",
        "Recall",
        "F1 Score"
    ])

    writer.writerow([
        precision,
        recall,
        f1
    ])

print(f"Metrics saved to: {metrics_path}")

Appending to /content/CNNStarDetectCentroid/training/evaluation.py


In [ ]:
%cd /content/CNNStarDetectCentroid
!git status

/content/CNNStarDetectCentroid
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   training/evaluation.py

no changes added to commit (use "git add" and/or "git commit -a")


In [2]:
# 1. Move to base directory
%cd /content

# 2. Clone your repository (Replace YOUR_GITHUB_USERNAME if different)
!git clone https://github.com/MMASirOwlbie/CNNProject-StarDetectionReproduction.git CNNStarDetectCentroid

# 3. Move into the repository directory
%cd /content/CNNStarDetectCentroid

# 4. Check git status to confirm you are inside the repo
!git status

/content
Cloning into 'CNNStarDetectCentroid'...
remote: Enumerating objects: 396, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 396 (delta 6), reused 5 (delta 1), pack-reused 377 (from 1)
Receiving objects: 100% (396/396), 232.64 MiB | 25.16 MiB/s, done.
Resolving deltas: 100% (106/106), done.
/content/CNNStarDetectCentroid
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
